# Machine Learning Workbenches: Environment Inspection, Execution, and Reproducibility

**Session:** 2026-10-31 · Competition tasks and metrics  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

Competitive machine learning and real-world AI pipelines require predictable, reproducible runtime environments. Whether running in a local virtual environment or an interactive cloud workbench, machine learning practitioners must be able to inspect hardware resources, set global random seeds, execute external scripts programmatically, and benchmark memory and time consumption.

## Learning Outcomes

- Inspect host system properties, Python interpreter versions, CPU core counts, and memory allocations.
- Implement a unified seed-setting protocol across standard libraries (`random`, `numpy`) for reproducible experiments.
- Execute external Python scripts and CLI commands programmatically using `subprocess.run`.
- Measure execution runtime using high-resolution monotonic clocks (`time.perf_counter()`).
- Monitor peak memory consumption with Python's built-in `tracemalloc` module.
- Encapsulate experiment hyperparameters and environment paths using Python `@dataclass`.


In [ ]:
import os
import platform
import sys

print(f"Operating System: {platform.system()} {platform.release()} ({platform.version()})")
print(f"Architecture:     {platform.machine()} ({platform.architecture()[0]})")
print(f"Python Version:   {sys.version.split()[0]} ({sys.executable})")
print(f"Logical CPU Count:{os.cpu_count()}")

## 1. Global Random Seed Management

Machine learning models, dataset splitters, and data augmentations rely on pseudo-random number generators (PRNGs). Failing to fix random seeds causes run-to-run variance, making it impossible to determine whether an improvement on a validation leaderboard is due to a better model architecture or simply random noise in initialization.


In [ ]:
import os
import random

import numpy as np


def seed_everything(seed: int = 42) -> None:
    """Fix random seeds across standard Python, environment, and NumPy."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    print(f"Global random seed set to {seed}.")


# Verify deterministic generation
seed_everything(42)
seq_a = [random.randint(1, 100) for _ in range(5)]
np_a = np.random.normal(0, 1, 4).round(3)

seed_everything(42)
seq_b = [random.randint(1, 100) for _ in range(5)]
np_b = np.random.normal(0, 1, 4).round(3)

assert seq_a == seq_b, "Python random generator is non-deterministic!"
assert np.array_equal(np_a, np_b), "NumPy random generator is non-deterministic!"
print("Reproducibility verified: identical sequences generated.")

## 2. Programmatic Script Execution with Subprocess

In modular AI workflows, standalone tasks (such as feature engineering, hyperparameter search, or external metric evaluators) are frequently invoked as separate processes. Using `subprocess.run` provides robust control over standard input/output, return codes, and error handling.


In [ ]:
import subprocess
import sys

# Execute a one-line Python script in a separate interpreter process
script_command = [
    sys.executable,
    "-c",
    "import math; print(f'Mathematical verification: pi={math.pi:.4f}, e={math.e:.4f}')",
]

result = subprocess.run(
    script_command,
    capture_output=True,
    text=True,
    check=True,
)

print(f"Process return code: {result.returncode}")
print(f"Process stdout:      {result.stdout.strip()}")

## 3. Profiling Execution Time and Memory Footprint

Understanding resource consumption is critical when choosing between algorithms under tight execution budgets. We combine `time.perf_counter()` for high-precision wall-clock timing with `tracemalloc` to record peak heap allocations.


In [ ]:
import time
import tracemalloc
from collections.abc import Generator
from contextlib import contextmanager

import numpy as np


@contextmanager
def benchmark_profile(tag: str) -> Generator[None, None, None]:
    """Context manager to measure elapsed wall time and peak memory allocation."""
    tracemalloc.start()
    start_time = time.perf_counter()
    try:
        yield
    finally:
        elapsed_sec = time.perf_counter() - start_time
        _current_mem, peak_mem = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        print(
            f"[{tag}] Elapsed: {elapsed_sec * 1000:.2f} ms | Peak RAM: {peak_mem / (1024 * 1024):.4f} MB"
        )


# Benchmark list comprehension vs NumPy vectorized allocation
with benchmark_profile("Python List Allocation (1M elements)"):
    py_list = [x * 2.5 for x in range(1_000_000)]

with benchmark_profile("NumPy Vectorized Allocation (1M elements)"):
    np_arr = np.arange(1_000_000, dtype=np.float64) * 2.5

## 4. Structuring Experiments with Configuration Dataclasses

Avoid scattering paths, hyperparameters, and switches across global variables. Encapsulating configuration in an immutable `@dataclass` ensures clean traceability and prevents accidental parameter mutations during training loops.


In [ ]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class WorkbenchConfig:
    experiment_id: str
    seed: int
    data_dir: Path
    output_dir: Path
    batch_size: int
    learning_rate: float
    use_stratified_cv: bool = True

    def summary(self) -> str:
        return (
            f"Experiment: {self.experiment_id} | Seed: {self.seed} | "
            f"Batch Size: {self.batch_size} | LR: {self.learning_rate} | "
            f"Stratified CV: {self.use_stratified_cv}"
        )


config = WorkbenchConfig(
    experiment_id="exp_baseline_01",
    seed=42,
    data_dir=Path("../data"),
    output_dir=Path("./outputs"),
    batch_size=32,
    learning_rate=0.001,
)

print(config.summary())
print("Configuration is frozen: dataclass ensures parameter immutability.")